In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# 1. Input data (Function 4)
# ============================================================
X_train = np.array([
    [0.89698105, 0.72562797, 0.17540431, 0.70169437],
 [0.8893564, 0.49958786, 0.53926886, 0.50878344],
 [0.25094624, 0.03369313, 0.14538002, 0.49493242],
 [0.34696206, 0.0062504, 0.76056361, 0.61302356],
 [0.12487118, 0.12977019, 0.38440048, 0.2870761],
 [0.80130271, 0.50023109, 0.70664456, 0.19510284],
 [0.24770826, 0.06044543, 0.04218635, 0.44132425],
 [0.74670224, 0.7570915, 0.36935306, 0.20656628],
 [0.40066503, 0.07257425, 0.88676825, 0.24384229],
 [0.6260706, 0.58675126, 0.43880578, 0.77885769],
 [0.95713529, 0.59764438, 0.76611385, 0.77620991],
 [0.73281243, 0.14524998, 0.47681272, 0.13336573],
 [0.65511548, 0.07239183, 0.68715175, 0.08151656],
 [0.21973443, 0.83203134, 0.48286416, 0.08256923],
 [0.48859419, 0.2119651, 0.93917791, 0.37619173],
 [0.16713049, 0.87655456, 0.21723954, 0.95980098],
 [0.21691119, 0.16608583, 0.24137226, 0.77006248],
 [0.38748784, 0.80453226, 0.75179548, 0.72382744],
 [0.98562189, 0.66693268, 0.15678328, 0.8565348],
 [0.03782483, 0.66485335, 0.16198218, 0.25392378],
 [0.68348638, 0.9027701, 0.33541983, 0.99948256],
 [0.17034731, 0.75695908, 0.27652049, 0.5312315],
 [0.85965692, 0.91959232, 0.20613873, 0.09779683],
 [0.28213837, 0.50598691, 0.53053084, 0.09630162],
 [0.32607578, 0.4723669, 0.453192, 0.10588734],
 [0.94838936, 0.89451301, 0.85163782, 0.55219629],
 [0.66495539, 0.04656628, 0.11677747, 0.79371778],
 [0.57776561, 0.42877174, 0.42582587, 0.24900741],
 [0.73861301, 0.48210263, 0.70936644, 0.50397001],
 [0.8548108, 0.49396462, 0.73530997, 0.80809201],
 [1.085621, 1.019592, 1.039177, 1.099482],
 [1.00000e-06, 1.00000e-06, 1.24558e-01, 1.00000e-06],
 [0.866175, 0.601115, 0.708072, 0.020585],
 [0.145904, 0.536548, 0.6014  , 0.01905 ],
 [0.356293, 0.442523, 0.13052 , 0.242559]
])

y_train = np.array([
    -22.10828779, -14.60139663, -11.69993246, -16.05376511, -10.06963343,
    -15.48708254, -12.68168498, -16.02639977, -17.04923465, -12.74176599,
    -27.31639636, -13.52764887, -16.6791152 , -16.50715856, -17.81799934,
    -26.56182083, -12.75832422, -19.44155762, -28.90327367, -13.70274694,
    -29.4270914 , -11.56574199, -26.85778644,  -7.96677535,  -6.70208925,
    -32.62566022, -19.98949793,  -4.02554228, -13.12278233, -23.1394284,
    -67.60493430274798, -22.782193418373407, -22.194212794446454, -13.363105653346768, -5.926020577803715
])

# ============================================================
# 2. Basic info: current best point (maximisation)
# ============================================================
current_best_idx = int(np.argmax(y_train))
current_best_x = X_train[current_best_idx]
current_best_y = float(y_train[current_best_idx])

print("Current best index:", current_best_idx)
print("Current best X:", current_best_x)
print("Current best y:", current_best_y)

# ============================================================
# 3. Normalise inputs to [0,1] (per feature) + standardise outputs
# ============================================================
X_min = X_train.min(axis=0)
X_max = X_train.max(axis=0)
X_scaled = (X_train - X_min) / (X_max - X_min + 1e-12)

y_mean = y_train.mean()
y_std = y_train.std() + 1e-12
y_scaled = (y_train - y_mean) / y_std

# ============================================================
# 4. Ensemble neural network surrogate
# ============================================================
class MLP(nn.Module):
    """
    Simple 2-hidden-layer MLP with dropout (for regularisation).
    We'll build an ensemble of these to approximate epistemic uncertainty.
    """
    def __init__(self, input_dim=4, hidden_dim=64, p_dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)

def train_single_model(model, X, y, n_epochs=800, lr=1e-3, weight_decay=1e-4):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    model.train()
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        preds = model(X)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        # Optional: print occasionally
        if (epoch + 1) % 200 == 0:
            print(f"[Model] Epoch {epoch+1}/{n_epochs} - MSE: {loss.item():.4f}")
    return model

# Seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=device)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32, device=device).unsqueeze(-1)

N_ENSEMBLE = 3
ensemble = []

print("\n=== Training ensemble surrogate ({} members) ===".format(N_ENSEMBLE))
for m in range(N_ENSEMBLE):
    print(f"\nTraining model {m+1}/{N_ENSEMBLE}")
    torch.manual_seed(100 + m)
    model = MLP(input_dim=4, hidden_dim=64, p_dropout=0.1).to(device)
    model = train_single_model(model, X_tensor, y_tensor,
                               n_epochs=800, lr=1e-3, weight_decay=1e-4)
    ensemble.append(model)

# ============================================================
# 5. Ensemble prediction: mean & std in ORIGINAL y-scale
# ============================================================
def ensemble_predict(ensemble, X):
    """
    X: torch tensor [N, D] in scaled space.
    Returns mean(N,) and std(N,) in ORIGINAL y units.
    """
    preds = []
    with torch.no_grad():
        for model in ensemble:
            model.eval()
            p_scaled = model(X).squeeze(-1)       # prediction in scaled y
            p_raw = p_scaled * y_std + y_mean     # back to original
            preds.append(p_raw)
    preds = torch.stack(preds, dim=0)  # [M, N]
    mean = preds.mean(dim=0)
    std = preds.std(dim=0) + 1e-9
    return mean, std

# Standard normal (for EI / PI)
normal = torch.distributions.Normal(
    torch.tensor(0.0, device=device),
    torch.tensor(1.0, device=device)
)

def expected_improvement(mean, std, best_y, xi=0.01):
    """
    EI for maximisation with normal approximation.
    mean, std: tensors in ORIGINAL y-scale.
    best_y: scalar (current best y in ORIGINAL scale).
    """
    imp = mean - best_y - xi
    Z = imp / std
    ei = imp * normal.cdf(Z) + std * torch.exp(normal.log_prob(Z))
    ei = torch.where(std > 0, ei, torch.zeros_like(ei))
    return ei

def probability_of_improvement(mean, std, best_y, xi=0.0):
    """
    PI = P(f(x) > best_y + xi)
    """
    imp = mean - best_y - xi
    Z = imp / std
    pi = normal.cdf(Z)
    return pi

# ============================================================
# 6. Propose next query point using EI + ensemble
# ============================================================
def propose_next_point(ensemble, X_min, X_max, n_candidates=5000):
    # Sample candidates uniformly in scaled [0,1]^4
    candidates_scaled = np.random.rand(n_candidates, 4).astype(np.float32)
    X_cand_tensor = torch.tensor(candidates_scaled, dtype=torch.float32, device=device)

    # Predict in original y-scale
    mean, std = ensemble_predict(ensemble, X_cand_tensor)

    best_y = current_best_y
    ei = expected_improvement(mean, std, best_y)
    pi = probability_of_improvement(mean, std, best_y)

    best_idx = int(torch.argmax(ei).item())
    next_x_scaled = candidates_scaled[best_idx]

    # Unscale back to original space
    next_x = X_min + next_x_scaled * (X_max - X_min)

    next_mean = float(mean[best_idx].item())
    next_std = float(std[best_idx].item())
    next_ei = float(ei[best_idx].item())
    next_pi = float(pi[best_idx].item())

    return next_x, next_mean, next_std, next_ei, next_pi

next_x, next_mean, next_std, next_ei, next_pi = propose_next_point(
    ensemble, X_min, X_max, n_candidates=5000
)

# ============================================================
# 7. Report: current best vs next query, probabilities & reasoning
# ============================================================
print("\n================ RESULTS ================")
print("Current best X:", current_best_x)
print("Current best y:", current_best_y)
print("\nProposed next query point X*:", next_x)
print("Predicted mean at X*:", next_mean)
print("Predictive std at X*:", next_std)
print("Expected Improvement at X*:", next_ei)
print("Probability of Improvement P(f(X*) > f_best):", next_pi)

# Human-readable reasoning
improvement_margin = next_mean - current_best_y
print("\nReasoning:")
print(f"- The ensemble surrogate predicts an output of {next_mean:.3f} at X*,")
print(f"  compared to the current best of {current_best_y:.3f}, so the expected gain is {improvement_margin:.3f}.")
print(f"- The predictive uncertainty (std) at X* is {next_std:.3f}.")
print(f"- Combining mean and uncertainty gives an Expected Improvement (EI) of {next_ei:.3f},")
print(f"  and a Probability of Improvement of {next_pi:.3f} (~{next_pi*100:.1f}% chance of beating the current best).")
print("- Because the ensemble uses multiple independently-trained MLPs,")
print("  the variance across models gives a more robust estimate of epistemic uncertainty than a single network.")


Current best index: 27
Current best X: [0.57776561 0.42877174 0.42582587 0.24900742]
Current best y: -4.02554228

=== Training ensemble surrogate (3 members) ===

Training model 1/3
[Model] Epoch 200/800 - MSE: 0.1156
[Model] Epoch 400/800 - MSE: 0.0498
[Model] Epoch 600/800 - MSE: 0.0414
[Model] Epoch 800/800 - MSE: 0.0313

Training model 2/3
[Model] Epoch 200/800 - MSE: 0.0810
[Model] Epoch 400/800 - MSE: 0.0588
[Model] Epoch 600/800 - MSE: 0.0334
[Model] Epoch 800/800 - MSE: 0.0319

Training model 3/3
[Model] Epoch 200/800 - MSE: 0.1606
[Model] Epoch 400/800 - MSE: 0.0882
[Model] Epoch 600/800 - MSE: 0.0736
[Model] Epoch 800/800 - MSE: 0.0482

================ RESULTS ================
Current best X: [0.57776561 0.42877174 0.42582587 0.24900742]
Current best y: -4.02554228

Proposed next query point X*: [0.06143152 0.38124752 0.98379276 0.7055758 ]
Predicted mean at X*: -17.341638565063477
Predictive std at X*: 4.552572727203369
Expected Improvement at X*: 0.0022449716925621033
Prob